# Brown dwarfs

This tutorial follows [Making a cluster](01_make_cluster.ipynb). That notebook builds an isochrone, an IMF, and a cluster. Here the evolution model is `MergedFlameSkimmerRedDiamondbackMIST`, which attaches metallicity-dependent Sonora brown dwarfs to MIST stars, and the atmosphere is `get_merged_atmosphere_v3`. The IMF extends into the brown-dwarf regime. Brown dwarfs have `phase` 90.

In [ ]:
# Import necessary packages.
from spisea import synthetic, evolution, atmospheres, reddening, ifmr
from spisea.imf import imf, multiplicity
import numpy as np
import pylab as py
import matplotlib.pyplot as plt
from astropy.table import vstack
%matplotlib inline


## Isochrone with Sonora brown dwarfs

The age is 10^8.4 yr, with no extinction and a distance of 10 pc, so the brown-dwarf sequence is easy to see. Cool objects with log(g) >= 2 use FlameSkimmer atmospheres.

In [ ]:
# Define isochrone parameters.
logAge = 8.4  # Age in log(years)
AKs = 0.0  # extinction in mags
dist = 10  # distance in parsec
metallicity = 0  # Metallicity in [M/H]

# Sonora brown dwarfs joined to MIST, and the recommended atmosphere.
evo_model = evolution.MergedFlameSkimmerRedDiamondbackMIST()
atm_func = atmospheres.get_merged_atmosphere_v3
red_law = reddening.RedLawHosek18b()
filt_list = ['wfc3,ir,f127m', 'wfc3,ir,f139m', 'wfc3,ir,f153m']
iso_dir = '_output/02_brown_dwarfs/'

my_iso = synthetic.IsochronePhot(logAge, AKs, dist, metallicity=metallicity,
                            evo_model=evo_model, atm_func=atm_func,
                            red_law=red_law, filters=filt_list,
                            iso_dir=iso_dir)


## Brown dwarfs on the isochrone

Masses below 0.08 solar masses are brown dwarfs in this model. Masses below 0.01 solar masses are not used here.

In [ ]:
# Brown dwarfs by mass, then by evolutionary phase.
bd_idx = np.where((my_iso.points['mass'] > 0.01) & (my_iso.points['mass'] < 0.08))[0]
phase_idx = np.where(my_iso.points['phase'] == 90)[0]
print('Brown dwarfs by mass: {0}'.format(len(bd_idx)))
print('Brown dwarfs by phase 90: {0}'.format(len(phase_idx)))
if len(bd_idx) > 0:
    f153m = np.round(my_iso.points[bd_idx[0]]['m_hst_f153m'], decimals=3)
    mass = my_iso.points[bd_idx[0]]['mass']
    print('Example brown dwarf: mass = {0:.3f} Msun, F153M = {1} mag'.format(mass, f153m))


In [ ]:
# Mass-magnitude diagram. Stars mark the brown-dwarf masses.
py.figure(1, figsize=(6, 6))
py.clf()
py.plot(my_iso.points['mass'], my_iso.points['m_hst_f153m'], 'r-', label='Isochrone')
py.plot(my_iso.points['mass'][bd_idx], my_iso.points['m_hst_f153m'][bd_idx],
        'b*', ms=8, label='Brown dwarf')
py.xlabel('Mass (solar masses)')
py.ylabel('F153M (mag)')
py.gca().invert_yaxis()
py.legend()
py.show()


## An IMF that includes brown dwarfs

The broken power law follows the Kirkpatrick slopes below 0.22 solar masses and the Salpeter and Kroupa slopes above that. Star systems are unresolved.

In [ ]:
imf_multi = multiplicity.MultiplicityUnresolvedOffner2023()

# The exponent includes the negative sign: dN/dm proportional to m^power.
massLimits = np.array([0.01, 0.05, 0.22, 0.55, 8, 120])
powers = np.array([-0.6, -0.25, -1.3, -2.3, -2.35])
my_imf = imf.IMF_broken_powerlaw(massLimits, powers, imf_multi)

cluster = synthetic.ResolvedCluster(my_iso, my_imf, 10**5.)
clust = cluster.star_systems
iso = my_iso.points
bds = iso['mass'] >= 0.01

py.figure(2, figsize=(8, 8))
py.clf()
py.plot(clust['m_hst_f127m'] - clust['m_hst_f153m'], clust['m_hst_f153m'],
        'k.', ms=5, alpha=0.1, label='Cluster')
py.plot(iso['m_hst_f127m'][bds] - iso['m_hst_f153m'][bds], iso['m_hst_f153m'][bds],
        'r-', label='Isochrone')
py.xlabel('F127M - F153M')
py.ylabel('F153M')
py.gca().invert_yaxis()
py.legend()
py.show()


## Brown dwarfs and compact objects

A larger cluster makes the rare compact remnants visible next to the brown dwarfs. `phase` is 103 for black holes, 102 for neutron stars, 101 for white dwarfs, and 90 for brown dwarfs.

In [ ]:
my_ifmr = ifmr.IFMR_Raithel18()
kc_imf = imf.Salpeter_Kirkpatrick_2024(multiplicity=imf_multi)
kc_cluster = synthetic.ResolvedCluster(my_iso, kc_imf, 10**6, ifmr=my_ifmr)
kc_out = kc_cluster.star_systems
kc_comp = kc_cluster.companions


In [ ]:
# Locate BHs, NSs, WDs, and BDs
p2_bh = np.where(kc_out['phase'] == 103)[0]
c_bh = np.where(kc_comp['phase'] == 103)[0]
p2_ns = np.where(kc_out['phase'] == 102)[0]
c_ns = np.where(kc_comp['phase'] == 102)[0]
p2_wd = np.where(kc_out['phase'] == 101)[0]
c_wd = np.where(kc_comp['phase'] == 101)[0]
p2_bd = np.where(kc_out['phase'] == 90)[0]
c_bd = np.where(kc_comp['phase'] == 90)[0]

# Define bins for histograms
bh_bins = np.linspace(14, 100, 20)
wd_bins = np.linspace(0.4, 10, 20)
bd_bins = np.linspace(0.01, 0.08, 8)
ns_bins = np.linspace(0, 30, 20)

# Create subplots
plt.figure(figsize=(14,6))

# Plot BHs
plt.subplot(2, 2, 1)
plt.hist(kc_out[p2_bh]['mass'], histtype='step', bins=bh_bins, label='Primary Black Holes', color='blue')
plt.hist(kc_comp[c_bh]['mass'], histtype='step', bins=bh_bins, label='Companion Black Holes', color='orange')
plt.title("Black Holes by Mass")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot WDs
plt.subplot(2, 2, 2)
plt.hist(kc_out[p2_wd]['mass'], histtype='step', bins=wd_bins, label='Primary White Dwarves', color='blue')
plt.hist(kc_comp[c_wd]['mass'], histtype='step', bins=wd_bins, label='Companion White Dwarves', color='orange')
plt.title("White Dwarves by Mass")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot BDs
plt.subplot(2, 2, 3)
plt.hist(kc_out[p2_bd]['mass'], histtype='step', bins=bd_bins, label='Primary Brown Dwarves', color='blue')
plt.hist(kc_comp[c_bd]['mass'], histtype='step', bins=bd_bins, label='Companion Brown Dwarves', color='orange')
plt.title("Brown Dwarves by Mass")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot NSs
plt.subplot(2, 2, 4)
plt.hist(kc_out[p2_ns]['mass'], histtype='step', bins=ns_bins, label='Primary Neutron Stars', color='blue')
plt.hist(kc_comp[c_ns]['mass'], histtype='step', bins=ns_bins, label='Companion Neutron Stars', color='orange')
plt.title("Neutron Stars by Mass")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Adjust space between subplots
plt.subplots_adjust(hspace=0.5)

# Show the plots
plt.show()


Plotted above are the number of generated compact objects (black holes, white dwarfs, and neutron stars) against the generated number of brown dwarfs.

In [ ]:
# Locate BHs, NSs, WDs, and BDs
p2_bh = np.where(kc_out['phase'] == 103)[0]
c_bh = np.where(kc_comp['phase'] == 103)[0]
k_bh = vstack([kc_out[p2_bh], kc_comp[c_bh]])
p2_ns = np.where(kc_out['phase'] == 102)[0]
c_ns = np.where(kc_comp['phase'] == 102)[0]
k_ns = vstack([kc_out[p2_ns], kc_comp[c_ns]])
p2_wd = np.where(kc_out['phase'] == 101)[0]
c_wd = np.where(kc_comp['phase'] == 101)[0]
k_wd = vstack([kc_out[p2_wd], kc_comp[c_wd]])
p2_bd = np.where(kc_out['phase'] == 90)[0]
c_bd = np.where(kc_comp['phase'] == 90)[0]
k_bd = vstack([kc_out[p2_bd], kc_comp[c_bd]])

# Create subplots
plt.figure(figsize=(14,6))

# Plot BHs
plt.subplot(2, 2, 1)
plt.hist(k_bh['mass'], histtype='step', bins=bh_bins, label='Progenitor Black Hole Masses', color='blue')
plt.hist(k_bh['mass_current'], histtype='step', bins=bh_bins, label='Current Black Hole Masses', color='orange')
plt.title("Black Holes by Mass Over Time")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot WDs
plt.subplot(2, 2, 2)
plt.hist(k_wd['mass'], histtype='step', bins=wd_bins, label='Progenitor White Dwarf Masses', color='blue')
plt.hist(k_wd['mass_current'], histtype='step', bins=wd_bins, label='Current White Dwarf Masses', color='orange')
plt.title("White Dwarves by Mass Over Time")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot BDs
plt.subplot(2, 2, 3)
plt.hist(k_bd['mass'], histtype='step', bins=bd_bins, label='Progenitor Brown Dwarf Masses', color='blue')
plt.hist(k_bd['mass_current'], histtype='step', bins=bd_bins, label='Current Brown Dwarf Masses', color='orange')
plt.title("Brown Dwarves by Mass Over Time")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Plot NSs
plt.subplot(2, 2, 4)
plt.hist(k_ns['mass'], histtype='step', bins=ns_bins, label='Progenitor Neutron Star Masses', color='blue')
plt.hist(k_ns['mass_current'], histtype='step', bins=ns_bins, label='Current Neutron Stars Masses', color='orange')
plt.title("Neutron Stars by Mass Over Time")
plt.xlabel('Mass ($M_\odot$)')
plt.ylabel('Number')
plt.legend()

# Adjust space between subplots
plt.subplots_adjust(hspace=0.5)

# Show the plots
plt.show()


Above we have plotted the initial masses of compact objects (black holes, neutron stars, and white dwarfs) against their current masses. We have done the same for brown dwarfs. As expected, brown dwarf masses do not change over time.